#### Calico

Updated

In [ ]:
kubectl create -f https://raw.githubusercontent.com/projectcalico/calico/v3.32.0/manifests/v1_crd_projectcalico_org.yaml

In [ ]:
kubectl create -f https://raw.githubusercontent.com/projectcalico/calico/v3.32.0/manifests/tigera-operator.yaml

In [ ]:
cat <<EOF > custom-resources.yaml
apiVersion: operator.tigera.io/v1
kind: Installation
metadata:
  name: default
spec:
  variant: Calico
  calicoNetwork:
    bgp: Disabled
    ipPools:
    - name: default-ipv4-ippool
      cidr: 10.244.0.0/16
      blockSize: 26
      encapsulation: VXLAN
      natOutgoing: Enabled
      nodeSelector: all()
---
apiVersion: operator.tigera.io/v1
kind: APIServer
metadata:
  name: default
spec: {}
EOF

In [ ]:
kubectl create -f custom-resources.yaml

Helper

In [ ]:
kubectl get tigerastatus

In [ ]:
kubectl patch felixconfiguration default --type='merge' -p '{"spec":{"bpfExternalServiceMode":"Tunnel"}}'

In [ ]:
kubectl patch felixconfiguration default --type='merge' -p '{"spec":{"bpfKubeProxyIptablesCleanupEnabled":false}}'

In [ ]:
kubectl delete pods -n calico-system -l k8s-app=calico-node

In [ ]:
kubectl patch daemonset kube-proxy -n kube-system --type json -p='[{"op": "remove", "path": "/spec/template/spec/nodeSelector/operator.tigera.io~1disable-kube-proxy"}]'

---

For test

In [ ]:
kubectl get ippools.crd.projectcalico.org

For delete evry thing realted to calico

In [ ]:
kubectl delete ns tigera-operator --force --grace-period=0
kubectl patch ns tigera-operator -p '{"metadata":{"finalizers":[]}}' --type=merge
kubectl get crd | grep -E 'calico|tigera|projectcalico|networking.k8s.io' | awk '{print $1}' | xargs -r kubectl delete crd --force --grace-period=0
kubectl delete crd clusternetworkpolicies.policy.networking.k8s.io --force --grace-period=0
kubectl get clusterrole | grep tigera
kubectl delete clusterrole tigera-operator tigera-operator-secrets --ignore-not-found

kubectl get clusterrolebinding | grep tigera
kubectl delete clusterrolebinding tigera-operator --ignore-not-found
kubectl get crd | grep calico
kubectl get crd | grep tigera

---

- Wait for everything to be ready
- All pods must show 1/1 Running before continuing

In [ ]:

kubectl get pods -n calico-system -w 

---

#### After add all nodes by ansible

In [ ]:
# make this folder for Nodes
mkdir /etc/kubernetes/manifests/

In [ ]:
scp /etc/kubernetes/manifests/kube-vip.yaml root@172.16.6.67:/etc/kubernetes/manifests/
scp /etc/kubernetes/manifests/kube-vip.yaml root@172.16.6.62:/etc/kubernetes/manifests/

In [ ]:
# On ALL nodes (masters and workers)
cat > /etc/sysctl.d/99-calico-rpfilter.conf << 'EOF'
# Required for Calico VXLAN to work properly
net.ipv4.conf.all.rp_filter = 0
net.ipv4.conf.default.rp_filter = 0
net.ipv4.conf.vxlan.calico.rp_filter = 0
EOF

# Apply immediately
sysctl --system

# Verify on all nodes
sysctl net.ipv4.conf.all.rp_filter
sysctl net.ipv4.conf.vxlan.calico.rp_filter

In [ ]:
# Change the reverse path filtering from "Strict" (1) to "Loose" (2)
sudo sysctl -w net.ipv4.conf.all.rp_filter=2
sudo sysctl -w net.ipv4.conf.default.rp_filter=2

# Make it permanent across reboots
echo "net.ipv4.conf.all.rp_filter=2" | sudo tee -a /etc/sysctl.conf
echo "net.ipv4.conf.default.rp_filter=2" | sudo tee -a /etc/sysctl.conf
sudo sysctl -p

In [ ]:
cat <<EOF > /etc/sysctl.d/99-kubernetes.conf
net.ipv4.ip_forward = 1
net.ipv4.conf.all.rp_filter = 0
net.ipv4.conf.default.rp_filter = 0
EOF

In [ ]:
# Force the main interface configuration to match loose filtering
echo "net.ipv4.conf.ens192.rp_filter=2" | sudo tee -a /etc/sysctl.d/99-kubernetes.conf
echo "net.ipv4.conf.all.rp_filter=2" | sudo tee -a /etc/sysctl.d/99-kubernetes.conf
echo "net.ipv4.conf.default.rp_filter=2" | sudo tee -a /etc/sysctl.d/99-kubernetes.conf

# Reload the kernel settings right now
sudo sysctl --system

## `Copy the join command and run it`